In [ ]:
import numpy as np
import scipy.linalg

# Shared firmware convention:
#   x = [body angle error, body angular rate, reported flywheel speed]
#   u = commanded motor torque = -K @ x
# Signs below are expressed in the measured IMU/encoder coordinates.
g = 9.81
m_body = 1.2
m_wheel = 0.0949134
com_height = 0.1
wheel_inertia = 78e-6
body_inertia = (1 / 3) * m_body * com_height**2 + m_wheel * com_height**2
gravity_gain = (m_body + m_wheel) * g * com_height / body_inertia

A = np.array([
    [0, 1, 0],
    [gravity_gain, 0, 0],
    [gravity_gain, 0, 0],
])
B = np.array([
    [0],
    [1 / body_inertia],
    [(1 / wheel_inertia) + (1 / body_inertia)],
])

# Q is positive semidefinite and R must be positive definite.
Q = np.diag([500.0, 100.0, 500_000.0])
R = np.array([[1_000_000_000.0]])
P = scipy.linalg.solve_continuous_are(A, B, Q, R)
K = np.linalg.solve(R, B.T @ P)
applied_feedback = -K
closed_loop_poles = np.linalg.eigvals(A - B @ K)

assert np.all(np.linalg.eigvalsh(Q) >= 0)
assert np.all(np.linalg.eigvalsh(R) > 0)
assert np.all(closed_loop_poles.real < 0)

print("K for firmware u = -Kx:", K[0])
print("Applied state coefficients:", applied_feedback[0])
print("Closed-loop poles:", closed_loop_poles)
print("Single-axis Riccati P (normally full):\n", P)

# Two orthogonal, uncoupled axes produce a block-diagonal 2x6 gain matrix.
# P is full within each axis; cross-axis terms appear only when the model
# includes physical X/Y coupling or a non-diagonal cross-axis cost.
A_xy = scipy.linalg.block_diag(A, A)
B_xy = scipy.linalg.block_diag(B, B)
Q_xy = scipy.linalg.block_diag(Q, Q)
R_xy = scipy.linalg.block_diag(R, R)
P_xy = scipy.linalg.solve_continuous_are(A_xy, B_xy, Q_xy, R_xy)
K_xy = np.linalg.solve(R_xy, B_xy.T @ P_xy)
K_xy_expected = scipy.linalg.block_diag(K, K)
assert np.allclose(K_xy, K_xy_expected, rtol=1e-4, atol=2e-6)
print("Two-axis K:\n", K_xy)